# Noms du binome: 

# Prénoms du binome:

## Note: exporter le compte rendu basé sur le notebook au format pdf


# Entropie et génération de mots de passe
* L'objectif de ce tp est de continuer à se familiariser avec la notion d'entropie, mais aussi de comprendre le lien qu'il existe entre cette mesure informationelle et la sécurité d'un générateur (humain ou executable) de mots de passes
* Ainsi, nous proposons d'étudier l'entropie d'un tel générateur, et ce en fonction du modèle probabiliste considéré pour le modéliser (contruit à partir d'une lettre, de deux lettres, de 4 lettres, ...). A l'aide de tirrages aléatoires, nous estimerons également le temps moyen nécessaire pour trouver un mot de passe à partir de ce modèle.
* A la fin de ce TP, nous considérerons un générateur de mots de passe spécifique qui générera un mot de passe en prennant **4 lettres consécutives dans un texte** (sans se soucier des espaces). Ces lettres peuvent faire parti d'un ou de plusieurs mots consécutifs.
* Nous faisons l'hypothèse que le texte n'est composé que des 26 lettres de l'alphabet, sans majuscules ni accents

Nous chercherons aussi à comprendre (voir dernière question):
- les bonnes pratiques pour le défenseur, i.e. la personne cherchant à générer/construire un système de génération de mots de passe.
- les bonnes pratiques pour l'attaquant, i.e. la personne essayant de trouver le mot de passe.

**Il est important de commenter vos réponses, en utilisant des cellules markdown**


In [1]:
import numpy as np

from numpy import genfromtxt
from pandas import read_csv
import pandas as pd
import time

#### Modèle monogramme (une lettre) : le générateur génère des mots de passe à partir des occurences des monogrammes
* On récupére des données composées de [lettre,frequence d'apparition de la lettre] (voir fichier csv pour [comma-separated-value](https://en.wikipedia.org/wiki/Comma-separated_values)) 

In [2]:
monogramme = read_csv('monogramme.csv')
freq_mono = (monogramme['frequency']).values
letters_mono = (monogramme['letters']).values

Q: Quelles sont les 5 lettres les plus représentées ?

In [3]:
# TO DO
monogramme.sort_values(by="frequency", ascending=False).head(5)

,letters,frequency
0,E,0.1776
1,S,0.0823
2,A,0.0768
3,N,0.0761
4,T,0.0730


Ecrire une fonction qui calcule l'entropie à partir d'un vecteur constitué de probabilités empiriques (note, il est important de bien *gérer* le cas ou la probabilité est nulle).

In [4]:
def entropie(freq):
    freq = np.array(freq)
    mask = freq > 0
    freq = freq[mask]
    ent = np.sum(freq*(-np.log2(freq)))
    return ent

entropie([0,0.5,0.5])

np.float64(1.0)

Q: en utilisant ce modèle probabiliste pour générer un mot de passe, quelle est l'entropie d'un mot de passe de 8 lettres ?

In [5]:
# TO DO
ent_mdp = 8*entropie(freq_mono)
#les tirges de lettres sont independants, donc l'entropie du mdp est la somme des entropies des lettres
ent_mdp

np.float64(31.676242429778334)

Q: A l'aide de la fonction `np.random.choice()`, estimer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ? (note: ici le tirage n'est pas forcemment réaliste, car aléatoire, mais l'idée est surtout de mesurer le temps minimal nécessaire pour générer N mots de passes).

In [6]:
t = time.time()

# TO DO
for i in range(100000):
    word = np.random.choice(letters_mono, 8, p=freq_mono)
t_100000 = time.time() - t
print(t_100000, 's')


1.2116005420684814 s


Nous definissons l'"entropie du devin" G (guessing entropie) comme le **nombre moyen d'essais successif nécessaires pour trouver un mot de passe à partir de notre générateur**.
    On peut montrer que $G\geq 2^H/4+1$ où $H$ est l'entropie de la source (voir le papier Password_Entropy_and_Password_Quality.pdf )

Q: calculer le minorant de $G$ pour ce modèle

In [7]:
# TO DO
mino = (2**(ent_mdp))/4 + 1
mino

np.float64(857904864.6814457)

Q: combien de temps cela prendra-t-il pour trouver un mot de passe si l'on suppose qu'il est possible de prendre le générateur codé précédemment ? (en minutes)

In [8]:
# TO DO
temps = (mino*t_100000)/100000
temps_min = temps/60
print(f"{temps_min} min")

173.23966651520448 min


On propose maintenant d'utiliser un modèle plus évolué qui est construit à partir de la probabilité conjointe de deux lettres successives (bigramme)

In [9]:

bigramme = read_csv('bigramme.csv',keep_default_na=False)


print(bigramme.head(5))
freq_bi = (bigramme['frequency']).values

letters_bi = (bigramme['letters']).values


  letters  frequency
0      AA   0.000252
1      AB   0.001760
2      AC   0.003482
3      AD   0.001960
4      AE   0.000200


Q: Quelles sont les 5 couples de lettres les plus représentés ?

In [10]:
# TO DO
bigramme = bigramme.sort_values(by="frequency", ascending=False)
print(bigramme.head(5))

    letters  frequency
122      ES   0.023809
117      EN   0.021248
82       DE   0.019570
290      LE   0.018845
357      NT   0.017009


Q: en utilisant ce modèle probabiliste pour générer un mot de passe, quelle est l'entropie d'un mot de passe de 8 lettres ?

In [11]:
# TO DO
bi_ent = 4*entropie(freq_bi)
bi_ent

np.float64(30.14226404646422)

Q: Pourquoi cette entropie est-elle inférieure à celle du modèle construit sur des monogrammes ? Quelle propriété théorique de l'entropie peut justifier ce constat  ?

La baisse d'entropie s'explique par la **sous-additivité de l'entropie** :

$$H(X,Y) = H(X) + H(Y) - I(X;Y) \leq H(X) + H(Y)$$

car l'information mutuelle $I(X;Y) \geq 0$, avec égalité si et seulement si $X$ et $Y$ sont indépendantes.

Or en français, deux lettres successives ne sont pas indépendantes (par exemple, Q est presque toujours suivi de U) : $I(X;Y) > 0$. L'entropie d'un bigramme est donc strictement inférieure à deux fois l'entropie d'une lettre, et un mot de passe de 8 lettres construit à partir de 4 bigrammes a une entropie plus faible (≈ 30,1 bits) que celui construit à partir de 8 lettres indépendantes (≈ 31,7 bits). Le modèle bigramme capture une partie de la structure de la langue, ce qui rend les mots de passe plus prévisibles.

Q: A l'aide de la fonction `np.random.choice()`, calculer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ?

In [12]:
t = time.time()

# TO DO
for i in range(100000):
    word = np.random.choice(letters_bi, 4, p=freq_bi)
t_100000_2 = time.time() - t

print(t_100000_2, 's')


1.5133957862854004 s


Q: calculer le minorant de $G$ pour ce modèle

In [13]:
# TO DO
mino_2 = (2 ** (bi_ent)) / 4 + 1
mino_2

np.float64(296254956.7015476)

Q: combien de temps cela prendra-t-il pour trouver un mot de passe si l'on suppose qu'il est possible de prendre le générateur codé précédemment ? (en minutes)

In [14]:
# TO DO
temps_2 = (mino_2 * t_100000_2) / 100000
temps_min_2 = temps_2 / 60
print(f"{temps_min_2} min")


74.72516718971433 min


Q: **Modèle Uniforme:** si maintenant on change de stratégie et on tire aléatoirement chaque lettre de l'alphabet de façon uniforme, quelle est l'entropie de ce nouveau générateur ?

In [15]:
# TO DO
freq_unif = np.ones(26) * 1 / 26
ent_unif = 8*entropie(freq_unif)
ent_unif

np.float64(37.603517745128734)

Q: A l'aide de la fonction `np.random.choice()`, calculer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ?

In [16]:
# TO DO*
t = time.time()

# TO DO
for i in range(100000):
    word = np.random.choice(letters_mono, 8, p=freq_unif)
t_100000_3 = time.time() - t

print(t_100000_3, "s")

1.1887938976287842 s


Q: calculer le minorant de $G$ pour ce modèle

In [17]:
# TO DO
mino_unif = (2 ** (ent_unif)) / 4 + 1
mino_unif


np.float64(52206766144.99988)

Q: dans ce cas précis, quelle est la valeur exacte de $G$?

**Démonstration.**

*Cadre.* Notons $N = 26^8$ le nombre de mots de passe possibles de 8 lettres, et $w_1, \dots, w_N$ ces mots. Soit $X$ la variable aléatoire « mot de passe produit par le générateur uniforme ». Les lettres sont tirées indépendamment et uniformément, donc pour tout $i \in \{1, \dots, N\}$ :
$$P(X = w_i) = \left(\frac{1}{26}\right)^8 = \frac{1}{N}.$$

*Stratégie de l'attaquant.* L'attaquant choisit un ordre de test, c'est-à-dire une permutation $\sigma$ de $\{1, \dots, N\}$. Il teste $w_{\sigma(1)}$, puis $w_{\sigma(2)}$, etc., sans jamais retester un mot (sinon il gaspille un essai).

*Variable aléatoire.* Soit $T$ le nombre d'essais nécessaires pour trouver le mot de passe :
$$T = k \iff X = w_{\sigma(k)}, \qquad k \in \{1, \dots, N\}.$$
On a donc, pour tout $k \in \{1, \dots, N\}$ :
$$P(T = k) = P\big(X = w_{\sigma(k)}\big) = \frac{1}{N}.$$
$T$ suit la loi uniforme sur $\{1, \dots, N\}$, et ce quel que soit l'ordre $\sigma$.

*Espérance.*
$$G = E[T] = \sum_{k=1}^{N} k \, P(T = k) = \frac{1}{N} \sum_{k=1}^{N} k = \frac{1}{N} \cdot \frac{N(N+1)}{2} = \frac{N+1}{2}.$$

*Optimalité.* Pour une loi quelconque, $E[T] = \sum_k k \, P(X = w_{\sigma(k)})$ est minimale quand les mots sont testés par probabilité décroissante (inégalité de réarrangement : on associe les plus petits $k$ aux plus grandes probabilités). Dans le cas uniforme, toutes les probabilités sont égales, donc tous les ordres sont optimaux et donnent $E[T] = \frac{N+1}{2}$. C'est bien la valeur exacte de $G$ :
$$\boxed{G = \frac{26^8 + 1}{2} \approx 1{,}04 \times 10^{11}}$$

*Vérification avec la borne.* $H = 8 \log_2 26 = \log_2 N$, donc $2^H/4 + 1 = N/4 + 1$, et on a bien $G = \frac{N+1}{2} \geq \frac{N}{4} + 1$ (soit environ 2 fois le minorant).

*Remarque.* Si l'attaquant tirait ses essais au hasard **avec remise**, chaque essai réussirait avec probabilité $p = 1/N$, indépendamment des autres. $T$ suivrait alors une loi géométrique de paramètre $p$, d'espérance $1/p = N$, soit deux fois plus d'essais : cette stratégie n'est pas optimale.

In [ ]:
# TO DO
N = 26**8
G = (N + 1) / 2
print(G)
print("rapport G / minorant :", G / mino_unif)

104413532288.5
rapport G / minorant : 1.9999999999712728


Q: combien de temps cela prendra-t-il pour trouver un mot de passe en utilisant le générateur codé précédemment ? (en minutes)

In [19]:
# TO DO
temps_3 = (G * t_100000_3) / 100000
temps_min_3 = temps_3 / 60
print(f"{temps_min_3} min")

20687.695002405802 min


Q: implémenter une attaque pratique qui consiste à:
1. **pour le défenseur: (la personne qui génère le mot de passe)** tirer un mot de passe de 4 lettres consécutives à partir de ce texte de Victor Hugo (texteFrancais.txt) tiré des Misérables.  
2. **pour l'attaquant:** utiliser le modèle bigramme pour générer des mots de passe et minimiser le nombre d'essais. Pour cela on pourra :
    * dans un premier temps pré-calculer un **dictionnaire**, qui contriendra un des MdP générés classés dans l'ordre du plus probable au moins probable et qui ne contient pas de doublons 
    * dans un deuxième temps appeler ce dictionnaire pour comparer chacune de ses entrées au mot de passe généré.
3. Il faudra faire ces tests plusieurs fois afin de d'obtenir un nombre moyens d'appel au dictionnaire nécessaire
4. Il sera intéressant de comparer le nombre trouvé à la valeur de G (qui est une borne inférieure)
5. Question annexe: Par un simple calcul, si le générateur utilisé n'est plus ce générateur mais un générateur qui tire chaque lettre de façon équiprobable, rappeler la valeur de G. Comparer cette valeur avec la valeur trouvée en utilisant la stratégie "des 4 lettres consécutives".

In [ ]:
## 1. Fonction générant un mot de passe
def get_passwd():
    text_hugo = open("texteFrancais.txt", "r")
    str_hugo = str(text_hugo.read())

    # On remplace des lettres avec accent avec des lettres sans accent
    str_hugo = str_hugo.replace("Â", "A")
    str_hugo = str_hugo.replace("Ù", "U")
    str_hugo = str_hugo.replace("Ô", "O")
    size_txt = len(str_hugo)

    idx_rand = np.random.randint(size_txt - 4)
    # print(idx_rand)

    psswd = str_hugo[idx_rand : idx_rand + 4]
    return psswd

In [21]:
# 2. Construction du dictionnaire / tableau des mots de 4 lettres + fréquences
from itertools import product

bigrammes = bigramme["letters"].tolist()
freq = dict(zip(bigramme["letters"], bigramme["frequency"]))

# mot -> fréquence
d = {a + b: freq[a] * freq[b] for a, b in product(bigrammes, repeat=2)}

# tableau des mots triés par fréquence décroissante
tab_passwd = np.array(
    [m for m, f in sorted(d.items(), key=lambda x: x[1], reverse=True)]
)

print(tab_passwd)


['ESES' 'ESEN' 'ENES' ... 'YHJR' 'YHJW' 'YHYH']


In [22]:
# 3. Attaques sur 1000 mots de passes avec les bigrammes

nb_trial = 1000
vec_nb_trials = np.zeros(nb_trial)
tab_passwd = np.array(tab_passwd)

for i in range(nb_trial):
    mdp = get_passwd()
    idx = np.where(tab_passwd == mdp)[0]  # positions où ça correspond
    vec_nb_trials[i] = idx[0] + 1 if len(idx) else -1


In [23]:
# 4. Comparaison avec la borne inférieure pour G
nb_echecs = np.sum(vec_nb_trials < 0)
print("mots de passe absents du dictionnaire :", nb_echecs, "/", nb_trial)

vec_found = vec_nb_trials[vec_nb_trials > 0]
G_emp = vec_found.mean()
print("nombre moyen d'essais (empirique) :", G_emp)

# entropie d'un mot de 4 lettres = 2 bigrammes = bi_ent/2
mino_G = (2 ** (bi_ent/2)) / 4 + 1
print("minorant de G :", mino_G)
print("rapport :", G_emp / mino_G)

mots de passe absents du dictionnaire : 0 / 1000
nombre moyen d'essais (empirique) : 13407.527
minorant de G : 8607.029219412801
rapport : 1.5577415456845287


Le nombre moyen d'essais obtenu empiriquement est supérieur au minorant de $G$ (environ 1,5 à 1,7 fois selon les tirages), ce qui est cohérent avec la borne $G \geq 2^H/4 + 1$
Il reste néanmoins de l'ordre de $10^4$ essais seulement, ce qui est très faible.

In [ ]:
# 5. Générateur uniforme sur 4 lettres :
# N = 26**4 mots équiprobables, stratégie optimale sans remise : G = (N+1)/2
N4 = 26**4
G_unif_4 = (N4 + 1) / 2
print("G (uniforme, 4 lettres) :", G_unif_4)
print("nombre moyen d'essais (4 lettres consécutives du texte) :", G_emp)
print("rapport :", G_unif_4 / G_emp)

G (uniforme, 4 lettres) : 228488.5
nombre moyen d'essais (4 lettres consécutives du texte) : 13407.527
rapport : 17.041807933707684


Avec un générateur uniforme, il faut en moyenne $G = (26^4+1)/2 \approx 228\,000$ essais, contre environ $13\,000$ à $15\,000$ avec la stratégie « 4 lettres consécutives d'un texte » attaquée par le modèle bigramme : le mot de passe tiré d'un texte en français est donc environ **15 à 17 fois plus facile** à trouver. Prendre des lettres consécutives dans un texte en langage naturel est une mauvaise stratégie pour le défenseur.

## Conclusions 

Définir des bonnes pratiques pour le défenseur, i.e. la personne cherchant à concevoir un système de génération de mots de passe ? 

L'objectif du défenseur est de maximiser l'entropie du générateur, puisque le nombre moyen d'essais de l'attaquant est minoré par $2^H/4 + 1$ :
- tirer chaque caractère de façon **uniforme et indépendante** : c'est la loi qui maximise l'entropie, $H = L \log_2 |A|$ (où $L$ est la longueur et $|A|$ la taille de l'alphabet) ;
- augmenter la **longueur** $L$ du mot de passe : l'entropie croît linéairement, donc le nombre d'essais croît exponentiellement ;
- augmenter la **taille de l'alphabet** (majuscules, minuscules, chiffres, symboles) ;
- **éviter le langage naturel** (mots, prénoms, extraits de texte) : les lettres y sont corrélées, ce qui fait chuter l'entropie. Notre expérience montre qu'un mot de passe tiré d'un texte est environ 15 fois plus facile (ordre de grandeur) à trouver qu'un mot de passe uniforme de même longueur.

Définir des bonnes pratiques pour l'attaquant, i.e. la personne essayant de trouver le mot de passe ?

- tester les mots de passe **du plus probable au moins probable, sans doublon** : c'est la stratégie optimale qui minimise le nombre moyen d'essais
- exploiter les **bases de mots de passe ayant fuité** et les dictionnaires (prénoms, mots courants, motifs comme « mot + chiffres ») ;
- précalculer le dictionnaire trié une fois pour toutes, puis le réutiliser pour chaque attaque.

Si le défenseur n'a pas appliqué ces bonnes pratiques, le mot de passe est trouvé en un temps bien inférieur à celui d'une recherche exhaustive.

## Un peu de lecture
Cet article montre comment des hackers, à partir de leaks de bases de mots de passes, peuvent rapidement arriver à trouver le votre:
https://arstechnica.com/information-technology/2013/05/how-crackers-make-minced-meat-out-of-your-passwords/
